<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/Module2Gate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'telco-customer-churn' dataset.
Path to dataset files: /kaggle/input/telco-customer-churn


In [37]:
import pandas as pd
import numpy as np
import os

# The kagglehub dataset downloaded path is in the 'path' variable
# Let's locate the CSV file within that path
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if csv_files:
    csv_file_path = os.path.join(path, csv_files[0])
    df = pd.read_csv(csv_file_path)
    print(f"Loaded dataset from: {csv_file_path}")
else:
    raise FileNotFoundError(f"No CSV file found in {path}")

# Inspect the structural health of the data immediately
print("--- Data Shape ---")
print(df.shape)

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Structural Information ---")
df.info()

print("\n--- Statistical Summary ---")
print(df.describe())

Loaded dataset from: /kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
--- Data Shape ---
(7043, 21)

--- First 5 Rows ---
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Y

### Notes & Predictions: Three Filtering Methods
1. **Single Column Filter:** Subsetting the DataFrame by `df['Churn']` will reduce our shape from a 2D matrix to a 1D Series containing exactly the same number of rows as the original dataset.
2. **Boolean Mask (Two Conditions):** I am filtering for long-tenure, high-paying customers (`tenure > 24` and `MonthlyCharges > 70`). I predict this will significantly reduce the row count, isolating a subset of core, high-value premium users.
3. **Using `.loc` for Target Tracking:** This method isolates rows where `Churn == 'Yes'` while dropping unrelated columns, keeping only `customerID` and `Contract`. This should produce a lean two-column DataFrame of churned users.


In [38]:
# 1. Filter to a single column (e.g., looking at just Churn status)
churn_column = df['Churn']
print("1. Single Column Shape:", churn_column.shape)

# 2. Boolean mask with two conditions (e.g., long tenure AND high monthly charges)
# Note: Adjust column names if yours use different casing
mask = (df['tenure'] > 24) & (df['MonthlyCharges'] > 70)
filtered_mask_df = df[mask]
print(f"2. Boolean Mask Filtered Rows: {filtered_mask_df.shape[0]}")

# 3. .loc with a row condition and a specific column choice
# Filters for churned customers and grabs only their Contract type
loc_filtered = df.loc[df['Churn'] == 'Yes', ['customerID', 'Contract']]
print("3. .loc Filtered Shape:", loc_filtered.shape)


1. Single Column Shape: (7043,)
2. Boolean Mask Filtered Rows: 2230
3. .loc Filtered Shape: (1869, 2)



### Notes & Predictions: Derived Columns
1. **Arithmetic Column (`Avg_Charge_Per_Tenure_Month`):** Dividing `TotalCharges` by `tenure` calculates the exact historical monthly average spent by the customer.
2. **Missing Value Cause:** We predict missing values (`NaN`) will appear in this new column. This is not caused by dividing by zero, but rather because new customers with a `tenure` of 0 have blank/missing values for their `TotalCharges` in the raw Kaggle dataset.
3. **Conditional Logic (`Spend_Category`):** By setting the splitting threshold exactly at `df['MonthlyCharges'].median()`, `np.where` will divide our dataset into two almost perfectly equal halves of 'High Spenders' and 'Low Spenders'.



In [39]:
# Ensure TotalCharges is numeric for the arithmetic step
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')

# 1. Corrected Arithmetic Derived Column: Exact average charge per month of tenure
df['Avg_Charge_Per_Tenure_Month'] = df['TotalCharges'] / df['tenure']

# 2. Conditional Derived Column: Labeling high vs low spenders using np.where
df['Spend_Category'] = np.where(df['MonthlyCharges'] > df['MonthlyCharges'].median(), 'High Spender', 'Low Spender')

# 3. Verification Check: Confirm that 0-tenure rows are natively NaN in TotalCharges
is_zero_tenure_nan = df.loc[df['tenure'] == 0, 'TotalCharges'].isna().all()
print(f"Verification: Are all 0-tenure TotalCharges rows NaN? {is_zero_tenure_nan}")

print("\nDerived columns successfully created. Current Columns:")
print(df[['Avg_Charge_Per_Tenure_Month', 'Spend_Category']].head())


Verification: Are all 0-tenure TotalCharges rows NaN? True

Derived columns successfully created. Current Columns:
   Avg_Charge_Per_Tenure_Month Spend_Category
0                    29.850000    Low Spender
1                    55.573529    Low Spender
2                    54.075000    Low Spender
3                    40.905556    Low Spender
4                    75.825000   High Spender


* **Spend Category vs. Churn:**
  Our derived `Spend_Category` shows that **High Spenders** have a higher churn rate than Low Spenders in this dataset. This establishes a baseline correlation between higher monthly costs and customer turnover, though it does not explicitly tell us the behavioral reason behind it.

* **Median-Split Verification:**
  Because our code assigns values strictly greater than (` > `) the median to "High Spender," any customer whose monthly bill sits exactly on the median value defaults into the "Low Spender" category. As a result, the output will likely print a hair under 50% for High Spenders and slightly over 50% for Low Spenders.

In [40]:
#1 Spend Category vs Churn (Closing the loop)
print("\n--- 1. Spend Category vs Churn Breakdown ---")
display(pd.crosstab(df_merged['Spend_Category'], df_merged['Churn'], normalize='index') * 100)

#2 Verifying the Spend Category Median Split
print("\n--- 2. Verification: Spend Category Proportions ---")
display(df_merged['Spend_Category'].value_counts(normalize=True) * 100)


--- 1. Spend Category vs Churn Breakdown ---


Churn,No,Yes
Spend_Category,,
High Spender,64.807966,35.192034
Low Spender,82.086168,17.913832



--- 2. Verification: Spend Category Proportions ---


,proportion
Spend_Category,
Low Spender,50.09229
High Spender,49.90771


### Notes & Predictions: Groupby Summary
* **Aggregation Targets:** Grouping by `Contract` (Month-to-month, One year, Two year) will reduce thousands of rows down to just 3 clean summary rows.
* **Expected Trends:** I predict that **Two-year contracts** will display the highest `Average_Tenure` and the lowest churn indicators, while **Month-to-month contracts** will display the highest volume of total customers but lower average stability.


In [41]:
# Group by Contract type, run 3 separate aggregations, sort the index/values, and rename outputs
groupby_summary = (
    df.groupby('Contract')
    .agg(
        Total_Customers=('customerID', 'count'),
        Average_Tenure=('tenure', 'mean'),
        Max_Monthly_Charges=('MonthlyCharges', 'max')
    )
    .sort_values(by='Average_Tenure', ascending=False)
)

display(groupby_summary)


,Total_Customers,Average_Tenure,Max_Monthly_Charges
Contract,,,
Two year,1695,56.735103,118.75
One year,1473,42.044807,118.60
Month-to-month,3875,18.036645,117.45


### Notes & Predictions: Left Merge Verification
* **Merge Strategy:** Using a `how='left'` merge keeps our primary dataset completely intact while bringing in metadata fields from the smaller secondary reference table.
* **Row Count Prediction:** Because the secondary table contains completely unique keys for each `Contract` type, the row count before and after the merge is **predicted to be exactly identical**. A mismatch would signal structural duplication errors in our lookup data.


In [42]:
# Create a dummy small metadata table to simulate your second table
unique_contracts = df['Contract'].unique()
mock_partner_table = pd.DataFrame({
    'Contract': unique_contracts,
    'Contract_Discount_Pct': [0, 5, 15] if len(unique_contracts) == 3 else [0] * len(unique_contracts)
})

# Verify rows before
rows_before = df.shape[0]
print(f"Rows before merge: {rows_before}")

# Perform the left merge
df_merged = df.merge(mock_partner_table, on='Contract', how='left')

# Verify rows after
rows_after = df_merged.shape[0]
print(f"Rows after merge: {rows_after}")

if rows_before == rows_after:
    print("Verification Success: Row counts match perfectly.")
else:
    print("Warning: Row count mismatch! Check for duplicate keys in your second table.")


Rows before merge: 7043
Rows after merge: 7043
Verification Success: Row counts match perfectly.


In [43]:
print(mock_partner_table)
print(df_merged['Contract_Discount_Pct'].isna().sum())

         Contract  Contract_Discount_Pct
0  Month-to-month                      0
1        One year                      5
2        Two year                     15
0



### Notes & Predictions: Crosstab Analysis
* **Analytical Question:** *How heavily does the length of a customer's contract commitment influence their likelihood to churn?*
* **Hypothesis:** Month-to-month customers have no financial or contractual friction preventing them from leaving. Therefore, I predict the `pd.crosstab` percentages will reveal a massive skew: significantly higher churn percentages among month-to-month users compared to locked-in 1-year or 2-year customers.

Question: How does the type of contract affect customer churn rates?

In [44]:
# Generate a cross-tabulation table showing exact counts
cross_tab_counts = pd.crosstab(df_merged['Contract'], df_merged['Churn'])
print("--- Churn Counts by Contract Type ---")
display(cross_tab_counts)

# Generate a cross-tabulation normalized by row to see the percentage breakdown
cross_tab_pct = pd.crosstab(df_merged['Contract'], df_merged['Churn'], normalize='index') * 100
print("\n--- Churn Percentages (%) by Contract Type ---")
display(cross_tab_pct.round(2))


--- Churn Counts by Contract Type ---


Churn,No,Yes
Contract,,
Month-to-month,2220,1655
One year,1307,166
Two year,1647,48



--- Churn Percentages (%) by Contract Type ---


Churn,No,Yes
Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83


### Notes & Predictions: Multi-Variable Crosstab Analysis
* **Analytical Question:** *How do age (SeniorCitizen) and family support structures (Dependents) jointly influence the type of service contract a customer signs?*
* **Hypothesis:**
  1. Non-senior customers with dependents are likely families; we predict they will favor stable, predictable **Two-year contracts** to keep household utility costs fixed.
  2. Senior citizens living without dependents represent a more isolated demographic; we predict they will skew heavily toward flexible, lower-commitment **Month-to-month contracts** due to fixed incomes or changing lifestyle needs.


In [45]:
# Cross-tabulate SeniorCitizen and Dependents (rows) against Contract type (columns)
# 'SeniorCitizen' is usually coded as 0 or 1 in this dataset
family_contract_counts = pd.crosstab(
    index=[df_merged['SeniorCitizen'], df_merged['Dependents']],
    columns=df_merged['Contract']
)

print("--- Demographic Counts by Contract Type ---")
display(family_contract_counts)

# Normalize by index (row) to see the exact percentage distribution for each group
family_contract_pct = pd.crosstab(
    index=[df_merged['SeniorCitizen'], df_merged['Dependents']],
    columns=df_merged['Contract'],
    normalize='index'
) * 100

print("\n--- Demographic Percentages (%) by Contract Type ---")
display(family_contract_pct.round(2))


--- Demographic Counts by Contract Type ---


Contract                  Month-to-month  One year  Two year
SeniorCitizen Dependents                                    
0             No                    2328       778       776
              Yes                    740       505       774
1             No                     758       164       129
              Yes                     49        26        16


--- Demographic Percentages (%) by Contract Type ---


Contract                  Month-to-month  One year  Two year
SeniorCitizen Dependents                                    
0             No                   59.97     20.04     19.99
              Yes                  36.65     25.01     38.34
1             No                   72.12     15.60     12.27
              Yes                  53.85     28.57     17.58

### Notes & Predictions: Analyzing Trends and Correlated Variables

* **Payment Method vs. Churn:**
  The data shows that customers using **Electronic Check** experience a higher percentage of churn compared to other payment options. A possible explanation to test later is whether electronic check users are disproportionately tied to shorter-term, lower-commitment contract types, making it easier for them to leave.

* **Internet Service Type:**
  The table shows a higher churn rate among **Fiber-optic users** compared to DSL users. While fiber is a faster technology, it is also typically more expensive. We can investigate whether higher monthly costs or specific contract structures make this group more volatile.

* **Testing for Confounding Variables:**
  By cross-tabulating `InternetService` and `PaymentMethod` against `Contract`, we can see if these higher-churn segments overlap heavily with month-to-month terms. If they do, contract flexibility might be the primary underlying driver of the churn we are seeing across these groups.


In [46]:
# 1. Payment Method vs Churn
print("--- 1. Payment Method vs Churn Breakdown ---")
display(pd.crosstab(df_merged['PaymentMethod'], df_merged['Churn'], normalize='index') * 100)

# 2. Internet Service vs Churn
print("\n--- 2. Internet Service vs Churn Breakdown ---")
display(pd.crosstab(df_merged['InternetService'], df_merged['Churn'], normalize='index') * 100)

# 3. Deep Dive Check: Does Contract type explain the Fiber Optic and Electronic Check trends?
print("\n--- 3. Deep Dive: Contract Breakdown by Group ---")
print("\nInternet Service vs Contract Type:")
display(pd.crosstab(df_merged['InternetService'], df_merged['Contract'], normalize='index') * 100)

print("\nPayment Method vs Contract Type:")
display(pd.crosstab(df_merged['PaymentMethod'], df_merged['Contract'], normalize='index') * 100)



--- 1. Payment Method vs Churn Breakdown ---


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),83.290155,16.709845
Credit card (automatic),84.756899,15.243101
Electronic check,54.714588,45.285412
Mailed check,80.893300,19.106700



--- 2. Internet Service vs Churn Breakdown ---


Churn,No,Yes
InternetService,,
DSL,81.040892,18.959108
Fiber optic,58.107235,41.892765
No,92.595020,7.404980



--- 3. Deep Dive: Contract Breakdown by Group ---

Internet Service vs Contract Type:


Contract,Month-to-month,One year,Two year
InternetService,,,
DSL,50.516316,23.543990,25.939694
Fiber optic,68.733850,17.409561,13.856589
No,34.338139,23.853211,41.808650



Payment Method vs Contract Type:


Contract,Month-to-month,One year,Two year
PaymentMethod,,,
Bank transfer (automatic),38.147668,25.323834,36.528497
Credit card (automatic),35.676741,26.149803,38.173456
Electronic check,78.224101,14.672304,7.103594
Mailed check,55.397022,20.905707,23.697270


In [47]:
churn_flag = df_merged['Churn'].eq('Yes')

for col in ['PaymentMethod', 'InternetService']:
    table = df_merged.assign(churned=churn_flag).pivot_table(
        index=col, columns='Contract', values='churned', aggfunc='mean'
    ) * 100
    print(f"\nChurn rate (%) by {col} within each Contract type")
    display(table.round(1))


Churn rate (%) by PaymentMethod within each Contract type


Contract,Month-to-month,One year,Two year
PaymentMethod,,,
Bank transfer (automatic),34.1,9.7,3.4
Credit card (automatic),32.8,10.3,2.2
Electronic check,53.7,18.4,7.7
Mailed check,31.6,6.8,0.8



Churn rate (%) by InternetService within each Contract type


Contract,Month-to-month,One year,Two year
InternetService,,,
DSL,32.2,9.3,1.9
Fiber optic,54.6,19.3,7.2
No,18.9,2.5,0.8


## Observation
* **Electronic check:** The raw churn gap against the automatic and mailed methods was about 26 to 30 points. Inside the month to month column it falls to about 20 to 22 points (53.7% against 31.6% to 34.1%). So contract mix explains roughly a quarter to a third of the gap, and the rest remains. The pattern repeats in the other columns: 18.4% against 6.8% to 10.3% on one year, and 7.7% against 0.8% to 3.4% on two year.

* **Fiber optic:** Contract explains almost none of the gap. The raw difference from DSL was about 22.9 points, and within month to month it is still about 22.4 points (54.6% against 32.2%). The gaps on one year (19.3% against 9.3%) and two year (7.2% against 1.9%) also stay clear of DSL.

Contract itself is a very strong effect. Churn drops sharply from month to month to two year inside every group, which fits earlier prediction. It is simply not the whole story for these two segments, so the accurate wording is that contract type matters a lot, but electronic check and fiber customers carry extra churn risk that contract mix alone does not explain.

In [49]:
print(type(df['MonthlyCharges']))
print(type(df[['MonthlyCharges']]))

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
